# 🏆 Challenge #1: Classification with MLP and CNN**  

Fecha de entrega: Lunes 23 de Marzo de 2026. 11:59 pm  

In this challenge, you will work on **two independent classification tasks**:

- One using **Multilayer Perceptrons (MLPs)**  
- One using **Convolutional Neural Networks (CNNs)**  

---

📊 **Global scoring system**

Your final grade is based on a global score that combines both tasks:

$$
\text{Score} = 0.5 \times \text{Accuracy}_{MLP} + 0.5 \times \text{Accuracy}_{CNN}
$$

where:

* **Accuracy_{MLP}** = classification accuracy of the MLP model (a value between 0 and 1).

* **Accuracy_{CNN}** = classification accuracy of the CNN model (a value between 0 and 1).

⚠️ The score will be computed by the professor using the predictions you submit and the hidden test labels (which you do not have access to).

---

🧠 **Task 1: Classification with MLP**

You must:

- Build a classification model using **only MLP architecture**  
- Train and validate the model  
- Generate predictions on the test set  

---

🧠 **Task 2: Classification with CNN**

You must:

- Build a classification model using a **CNN architecture**  
- Train and validate the model  
- Generate predictions on the test set  

⚠️ Only the methods and tools covered in class are allowed.

---

📦 **Submission**

The student must submit:

- The **notebook** showing all the work performed.  
  - ⚠️ If the notebook is not submitted, the score will **not** be calculated and the final grade will be **0.0**.  
  - In case of suspected fraud, the notebook will be reviewed in detail to verify the work.

- The **`.npy` file** with the MLP classification model predictions (as indicated in the notebook).

- The **`.npy` file** with the CNN classification model predictions (as indicated in the notebook).

- The **`.keras` file** of the CNN classification model.


In [1]:
# Type your code and your group.
STUDENT_CODE = 2235503             # Example: 2208534
GROUP = "B2"                   # 'B1', 'B2', ...

In [126]:
# Import Libraries
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


In [127]:
print(tf.config.experimental.get_device_details(tf.config.list_physical_devices('GPU')[0])['device_name'])


NVIDIA GeForce RTX 4060


In [161]:
# Seeds already set in imports cell above
os.environ['PYTHONHASHSEED'] = '42'
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

<br><br>

## Classification with MLPs

At the Universidad Industrial de Santander (UIS), the Campus Analytics group is building a system to automatically label campus spots by their activity profile. Short sensor scans (pedestrian counters, noise meters, environmental probes, and anonymized connectivity signals) are collected at specific points around campus (offices, classrooms, cafeterias, study areas, walkways). Each scan must be assigned to one of five activity categories so facility managers can plan cleaning, safety, ventilation, and space allocation more efficiently.

**Task**

Train a Multilayer Perceptron (MLP) that, given a tabular feature vector for each scan, predicts one of five classes:

*  0 — Quiet study area (low pedestrian flow, low noise, stable conditions).
*  1 — Classroom in session (moderate pedestrian flow, structured noise patterns).
*  2 — Office/admin (low–moderate flow, consistent noise, high enclosure).
*  3 — Cafeteria peak (high pedestrian flow, high noise variability, strong periodic bursts).
*  4 — Outdoor walkway (medium–high pedestrian flow, higher ventilation/dispersion).

The label y in the datasets is an integer in **{0,1,2,3,4}.**


**Features (columns):**

Each row corresponds to a short time window measurement at a single campus spot. All features are numeric:

*  **x0:** estimated pedestrian density (people/min).
*  **x1:** vehicular or bicycle flow near the spot (units/min).
*  **x2:** shade/solar exposure index (0–1).
*  **x3:** noise variability within the window (std/normalized).
*  **x4:** interaction between pedestrian & vehicular flow (normalized product).
*  **x5:** enclosure ratio (building height / open space proxy).
*  **x6:** social vitality from anonymized connectivity signals (0–1).
*  **x7:** green coverage ratio within ~50 m (0–1).
*  **x8:** periodicity of pedestrian flow (dominant frequency, normalized).
*  **x9:** pollutant/particulate dispersion proxy (higher → more dispersion/ventilation).

You will also see an id column used only as a row identifier. **Do not use it as input to your model**.

**Objective**

*  **Target variable:** y = Campus Activity Category (integer in {0,1,2,3,4}).
*  **Task:** Train a Multilayer Perceptron (MLP) to classify each measurement window into one of five campus activity categories, given the features x0..x9.
*  **Evaluation metric:** Accuracy is the official metric (higher is better).

In [162]:
# Load your training data
df_train_cls = pd.read_csv(f'{GROUP}_classification_train.csv')
df_train_cls = df_train_cls.drop(columns=['id'])

y_train_cls = df_train_cls['y']
x_train_cls = df_train_cls.drop(columns=['y'])

print(f"Number of training samples: {len(x_train_cls)}")

Number of training samples: 5000


In [163]:
# Split manual con shuffle
x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_cls, y_train_cls,
    test_size=0.2, random_state=42,
)

# Escalar
scaler = StandardScaler()
x_train_cls_scaled = scaler.fit_transform(x_tr)
x_val_scaled = scaler.transform(x_val)

In [172]:
# Create your model
model_mlp = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train_cls.shape[1],)),    
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(5, activation='softmax')
])
model_mlp.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model_mlp.fit(
    x_train_cls_scaled, y_tr,
    epochs=40,
    batch_size=64,
    validation_data=(x_val_scaled, y_val),
    verbose=1
)

Epoch 1/40


I0000 00:00:1774323104.708369   27338 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1067160__.13


55/63 ━━━━━━━━━━━━━━━━━━━━ 0s 928us/step - accuracy: 0.4181 - loss: 1.4513

I0000 00:00:1774323105.489088   27335 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1067160__.13


63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - accuracy: 0.6655 - loss: 1.0966 - val_accuracy: 0.8840 - val_loss: 0.5787
Epoch 2/40
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8855 - loss: 0.4325 - val_accuracy: 0.8970 - val_loss: 0.3571
Epoch 3/40
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8907 - loss: 0.3465 - val_accuracy: 0.8970 - val_loss: 0.3412
Epoch 4/40
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8913 - loss: 0.3350 - val_accuracy: 0.8990 - val_loss: 0.3380
Epoch 5/40
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8928 - loss: 0.3307 - val_accuracy: 0.9000 - val_loss: 0.3370
Epoch 6/40
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8942 - loss: 0.3284 - val_accuracy: 0.9000 - val_loss: 0.3367
Epoch 7/40
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8945 - loss: 0.3267 - val_accuracy: 0.8990 - val_loss: 0.3366
Epoch 8/40
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8957 - loss: 0.3253 - val_accuracy: 0.8980 - val_loss: 0.3366
Ep

In [ ]:
# FEEL FREE TO ADD MORE CELLS, CODE, OR ANALYSIS WHEREVER YOU NEED IT, BUT KEEP IT ORGANIZED.



In [173]:
# Load your test data
df_test_cls = pd.read_csv(f'{GROUP}_classification_test_student.csv')
x_test_cls = df_test_cls.drop(columns=['id'])
x_test_cls_scaled = scaler.transform(x_test_cls)

print(f"Number of testing samples: {len(x_test_cls)}")

Number of testing samples: 1500


In [174]:
# Use model_mlp.predict to get the predictions
preds_mlp = model_mlp.predict(x_test_cls_scaled)

47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [175]:
# Check if the size is (1500, 5)
if preds_mlp.shape == (1500, 5):
    np.save(f"{STUDENT_CODE}_classification.npy", preds_mlp)
    print("✅ The predictions have been saved. Do not forget to download them.")
else:
    print(f"⚠️ Predictions size is not correct: {preds_mlp.shape}, the file have not been saved.")

✅ The predictions have been saved. Do not forget to download them.


---
## **3. Pulmonary Nodules Classification** <a name="CNN_nodule_classification"></a>


A very important task is the automatic detection and classification of pulmonary nodules, in order to assist radiologists in these tasks. In the case of classification, it is very important to identify each pulmonary nodule, as well as to characterise and quantify the texture, size (given that it ranges from 3-30 mm) and morphology of the nodules.

<center>
<img src="https://media.springernature.com/lw685/springer-static/image/art%3A10.1007%2Fs11547-019-01130-9/MediaObjects/11547_2019_1130_Fig1_HTML.png" width="1000">
</center>



---

In [ ]:
#@title **Load** your training data
X_train = np.load(f'/content/{GROUP}_nodules_data_train.npy')
y_train = np.load(f'/content/{GROUP}_nodules_labels_train.npy')

print(f'This dataset contains {X_train.shape[0]} images of {X_train.shape[1:4]} pixels')
print(f'And {y_train.shape[0]} binary labels')

In [ ]:
#@title **CODE:** It is time to show some pulmonary nodules examples...

plt.figure(figsize=(15,15))
plt.subplot(441), plt.imshow(X_train[1], cmap = 'gray'), plt.xticks([]), plt.yticks([]);
plt.subplot(442), plt.imshow(X_train[2], cmap = 'gray'), plt.xticks([]), plt.yticks([]);
plt.subplot(443), plt.imshow(X_train[3], cmap = 'gray'),plt.xticks([]), plt.yticks([]);
plt.subplot(444), plt.imshow(X_train[8], cmap = 'gray'),plt.xticks([]), plt.yticks([]);
plt.subplot(445), plt.imshow(X_train[50], cmap = 'gray'), plt.xticks([]), plt.yticks([]);
plt.subplot(446), plt.imshow(X_train[100], cmap = 'gray'), plt.xticks([]), plt.yticks([]);
plt.subplot(447), plt.imshow(X_train[200], cmap = 'gray'),plt.xticks([]), plt.yticks([]);
plt.subplot(448), plt.imshow(X_train[500], cmap = 'gray'),plt.xticks([]), plt.yticks([]);

In [ ]:
#@title Do all the **pre-processing** that you consider.

"""
Put your code here
"""

In [ ]:
#@title **Implement** the CNN with your desired configuration. The input image size **MUST** be $224 \times 224 \times 1$

"""
model_conv = 
Put your code here
"""

In [ ]:
#@title **Compile** the model with the optimizer, loss and desired metrics.

"""
Put your code here
"""

In [ ]:
#@title **Train and predict** your model

"""
Put your code here
"""

In [ ]:
#@title Save your **.keras model**.
model_conv.save(f'{STUDENT_CODE}_ClasNod_Model.keras')

In [ ]:
#@title **Plot** the **training** and the **validation** loss.

"""
Put your code here
"""

In [ ]:
#@title Compute your **metrics** such as: confusion matrix, precision, recall, accuracy and AUC-ROC.

"""
Put your code here
"""

In [ ]:
#@title **Load** your test data
X_test = np.load(f'/content/{GROUP}_nodules_data_test.npy')
print(f'This dataset contains {X_test.shape[0]} images of {X_test.shape[1:4]} pixels')

In [ ]:
#@title Use **model_cls.predict** to get the **predictions**
preds_cls = ...

In [ ]:
# Check if the size is (749, 2)
if preds_cls.shape == (749, 2):
    np.save(f"{STUDENT_CODE}_classification.npy", preds_cls)
    print("✅ The predictions have been saved. Do not forget to download them.")
else:
    print(f"⚠️ Predictions size is not correct: {preds_cls.shape}, the file have not been saved.")

**IMPORTANTE:** En caso de quedar en el top 3 del challenge, se le solicitará en clase una explicación de la solución para que todos aprendamos 🙏 🛐.

<br><br><br><br>